## Filter NACC data into cases with all correct answers, all incorrect answers and the ones in between

Created by: Sahana Kowshik

Date: 05/21/2025

In [2]:
import pandas as pd
import ast
import re

In [3]:
train_data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv")
data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/llm_answer_extractor/extracted_results_4/Train/qwen25_3b_with_option_shuffling_Train.csv")

In [4]:
data.head()

,problem,prompt,generated_text,finish_reason,ID,Q_TYPE,UNQ_ID,prediction,keys,group,extraction_type,ground_truth
0,"{'ID': 'NACC000011', 'patient_summary': '{\n ...",<|im_start|>system\nYou are a helpful AI assis...,"<think>\nTo determine the correct MCI subtype,...",stop,NACC000011,MCI subtype,172095,E,"['A', 'B', 'C', 'D', 'E']",valid,regex,D
1,"{'ID': 'NACC000011', 'patient_summary': '{\n ...",<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, let's summarize the patient's ...",stop,NACC000011,MCI subtype,172094,A,"['A', 'B', 'C', 'D', 'E']",valid,regex,D
2,"{'ID': 'NACC000011', 'patient_summary': '{\n ...",<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, I need to provide a summary of...",stop,NACC000011,MCI subtype,172093,D,"['A', 'B', 'C', 'D', 'E']",valid,regex,D
3,"{'ID': 'NACC000011', 'patient_summary': '{\n ...",<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, I will analyze the patient's b...",stop,NACC000011,MCI subtype,172092,D,"['A', 'B', 'C', 'D', 'E']",valid,regex,D
4,"{'ID': 'NACC000011', 'patient_summary': '{\n ...",<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, I will construct a summary of ...",stop,NACC000011,MCI subtype,172091,A,"['A', 'B', 'C', 'D', 'E']",valid,regex,D


In [5]:
train_data.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text,Q_TYPE,visit_summary
0,NACC335873,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...",Identify the primary etiology of the patient's...,"A. Prion disease (CJD, other)\nB. Frontotempor...",H,Not applicable (no cognitive impairment),Primary etiology,The patient is a 67-year-old right-handed Whit...
1,NACC753667,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...",Identify the patient's cognitive status based ...,A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,B,Normal Cognition (NC),Cognitive status,The patient is a 74-year-old right-handed fema...
2,NACC229602,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","From the given patient information, select the...",A. Systemic and environmental factors includin...,J,Not applicable (no cognitive impairment),Primary etiology,The patient is a 94-year-old right-handed male...
3,NACC793943,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","Based on the clinical data, identify the neuro...",A. AD and FTLD\nB. AD and LBD and FTLD\nC. Alz...,D,AD and LBD,Neuropath,The subject is a 70-year-old male who lives wi...
4,NACC086845,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...",Determine the correct subtype of mild cognitiv...,A. Amnestic MCI- single domain\nB. Amnestic MC...,C,Not applicable (no diagnosis of MCI),MCI subtype,The subject is a 66-year-old married male who ...


In [6]:
print(train_data[train_data['Q_TYPE'] == "Primary etiology"].iloc[0]['options'])

A. Prion disease (CJD, other)
B. Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)
C. Alzheimer's disease (AD)
D. Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)
E. Normal-pressure hydrocephalus (NPH)
F. Vascular brain injury or vascular dementia including stroke (VD)
G. Lewy body disease (LBD)
H. Not applicable (no cognitive impairment)
I. Traumatic brain injury (TBI)
J. Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)


In [7]:
print(len(data['ID'].unique()))
print(len(train_data))

44541
44541


In [8]:
def pet_summary_with_invalid_keys(df):
    invalid = []
    sub_df = df[df['Q_TYPE'] == 'Amyloid PET'].reset_index(drop=True)
    for i, row in sub_df.iterrows():
        if "amyloid" in row['visit_summary'].lower():
            invalid.append(row['ID'])
            
    return invalid

def dat_summary_with_invalid_keys(df):
    invalid = []
    sub_df = df[df['Q_TYPE'] == 'DATSCAN'].reset_index(drop=True)
    for i, row in sub_df.iterrows():
        if "datscan" in row['visit_summary'].lower():
            invalid.append(row['ID'])
            
    return invalid

In [9]:
invalid_ids_pet = pet_summary_with_invalid_keys(train_data)
invalid_ids_dat = dat_summary_with_invalid_keys(train_data)

invalid_df = train_data[(train_data['ID'].isin(invalid_ids_pet)) | (train_data['ID'].isin(invalid_ids_dat))]
len(invalid_df)

8

In [10]:
train_data = train_data[~train_data['ID'].isin(invalid_df['ID'])].reset_index(drop=True)

In [11]:
print(len(data['ID'].unique()))
print(len(train_data))

44541
44533


In [12]:
train_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary_without_invalid.csv", index=False)

In [14]:
len(data[data['extraction_type'] == 'regex'])

328459

In [15]:
def modify(results_df):
    results_combined = results_df.copy()[['prediction','ground_truth', 'ID']]
    results_combined['prediction'] = results_combined['prediction']
    results_combined['correctness'] = results_combined['prediction'] == results_combined['ground_truth']
    results_combined = results_combined.reset_index(names=['problem']).reset_index(drop=True)
    return results_combined

In [16]:
modified_data = modify(data)

In [17]:
summary_df = modified_data.groupby('ID').agg(
    group_size=('correctness', 'size'),
    num_correct=('correctness', 'sum')  # True counts as 1
).reset_index()

In [18]:
modified_data[modified_data['ID'] == 'NACC000034']

,problem,prediction,ground_truth,ID,correctness
8,8,B,A,NACC000034,False
9,9,A,A,NACC000034,True
10,10,B,A,NACC000034,False
11,11,A,A,NACC000034,True
12,12,A,A,NACC000034,True
13,13,B,A,NACC000034,False
14,14,B,A,NACC000034,False
15,15,A,A,NACC000034,True


In [19]:
summary_df

,ID,group_size,num_correct
0,NACC000011,8,5
1,NACC000034,8,4
2,NACC000067,8,6
3,NACC000073,8,5
4,NACC000095,8,7
...,...,...,...
44536,NACC999854,8,0
44537,NACC999872,8,0
44538,NACC999922,8,8
44539,NACC999954,8,7


In [20]:
all_correct = summary_df[summary_df['num_correct'] == 8].reset_index(drop=True)
all_incorrect = summary_df[(summary_df['num_correct'] == 0) & (summary_df['group_size'] == 8)].reset_index(drop=True)
invalid_group = summary_df[summary_df['group_size'] != 8].reset_index(drop=True)
remaining = summary_df[
    (~summary_df['ID'].isin(all_correct['ID'])) &
    (~summary_df['ID'].isin(all_incorrect['ID'])) &
    (~summary_df['ID'].isin(invalid_group['ID']))
].reset_index(drop=True)

In [21]:
len(invalid_group)

0

In [22]:
remaining_data = train_data[train_data['ID'].isin(remaining['ID'])].reset_index(drop=True)
all_correct_data = train_data[train_data['ID'].isin(all_correct['ID'])].reset_index(drop=True)
all_incorrect_data = train_data[train_data['ID'].isin(all_incorrect['ID'])].reset_index(drop=True)

In [23]:
# print(f"Number of invalid cases: {len(invalid_data)}")
print(f"Number of all correct cases: {len(all_correct_data)}")
print(f"Number of all incorrect cases: {len(all_incorrect_data)}")
print(f"Number of selected cases: {len(remaining_data)}")

Number of all correct cases: 10698
Number of all incorrect cases: 7442
Number of selected cases: 26393


In [24]:
assert len(remaining_data) + len(all_correct_data) + len(all_incorrect_data) == len(train_data)

In [25]:
len(train_data)

44533

In [27]:
remaining_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/select_train_cases/train_selected.csv", index=False)
all_correct_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/select_train_cases/all_correct.csv", index=False)
all_incorrect_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/select_train_cases/all_incorrect.csv", index=False)
invalid_df.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/select_train_cases/invalid.csv", index=False)

In [28]:
pet_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

In [29]:
for key in pet_keys_remove:
    for i, row in remaining_data[remaining_data['Q_TYPE'] == "Amyloid PET"].iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [30]:
dat_keys_remove = [
    'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
]

In [31]:
for key in dat_keys_remove:
    for i, row in remaining_data[remaining_data['Q_TYPE'] == "DATSCAN"].iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [32]:
test = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/test_summary.csv")

/scratch/8940871.1.cds-gpu-long/ipykernel_3504039/3908899712.py:1: DtypeWarning: Columns (20,22,24,28,43,45,50,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,155,164,175,178,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,398,400,420,422,431,444,453,571,599,607,663,679,696,699,716,727,733,808,819,825,892,948,949,957,958,959,960,998,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation

In [33]:
set(test['NACCID']).intersection(set(train_data['ID']))

set()